# **IMPORT LIBRARIES**

In [1]:
import pandas as pd
import matplotlib.pyplot as plt


# **READ CSV**

In [ ]:
# load IMDb data
titles = pd.read_csv("title.basics.tsv", sep='\t')
ratings = pd.read_csv("title.ratings.tsv", sep='\t')
akas = pd.read_csv("title.akas.tsv", sep='\t')
crew = pd.read_csv("title.crew.tsv", sep='\t')
name_basic = pd.read_csv("name.basics.tsv", sep='\t')
principal = pd.read_csv("title.principals.tsv", sep='\t')


In [ ]:
name_basic

,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
0,nm0000001,Fred Astaire,1899,1987,"actor,miscellaneous,producer","tt0072308,tt0050419,tt0027125,tt0025164"
1,nm0000002,Lauren Bacall,1924,2014,"actress,miscellaneous,soundtrack","tt0037382,tt0075213,tt0038355,tt0117057"
2,nm0000003,Brigitte Bardot,1934,\N,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"
3,nm0000004,John Belushi,1949,1982,"actor,writer,music_department","tt0072562,tt0077975,tt0080455,tt0078723"
4,nm0000005,Ingmar Bergman,1918,2007,"writer,director,actor","tt0050986,tt0069467,tt0083922,tt0050976"
...,...,...,...,...,...,...
14872968,nm9993714,Romeo del Rosario,\N,\N,"animation_department,art_department","tt11657662,tt14069590,tt2455546"
14872969,nm9993716,Essias Loberg,\N,\N,\N,\N
14872970,nm9993717,Harikrishnan Rajan,\N,\N,cinematographer,tt8736744
14872971,nm9993718,Aayush Nair,\N,\N,cinematographer,tt8736744


In [22]:
# lọc top 250 movies
top_movies = (
    ratings.merge(titles[['tconst', 'titleType']], on='tconst')
           .query("titleType == 'movie'")
           .sort_values('averageRating', ascending=False)
           .head(250)
           .reset_index(drop=True)
)





In [31]:
top_movies_country = top_movies.merge(akas[['titleId','region']], left_on='tconst', right_on='titleId')
top_movies_country = top_movies_country[top_movies_country['region']!= r'\N']
top_movies_country_count = top_movies_country.groupby('region').size().reset_index(name='num_top_movies')
top_movies_country_count["weight"] = top_movies_country_count["num_top_movies"] / top_movies_country_count["num_top_movies"].max()
top_movies_country_count = top_movies_country_count.sort_values('weight', ascending=False) \
                                                   .reset_index(drop=True)

In [34]:
df_movie = pd.read_csv('movies.csv')
df_actor = pd.read_csv('actors_final.csv')
df_director = pd.read_csv('directors_final.csv')

In [36]:
df_actor

,actor_id,actor_name,birth_year,nationality,num_movies_acted
0,e4d693f2-7e6e-4a2e-9cd5-da21081ea3e6,Leonardo DiCaprio,1976,Eritrea,88
1,968cb23f-39ea-4d8a-8b68-45623dc0a5a6,Morgan Freeman,1994,Honduras,112
2,77359e10-64d8-4655-94d1-3b3a4307020c,Brad Pitt,1994,Tajikistan,36
3,97c1cb2e-09fd-4846-992a-a69396c2e732,Tom Hanks,1951,Somalia,65
4,fd23ffb5-57b7-4faf-8212-36e720e50d77,Robert Downey Jr.,1987,Turkey,94
...,...,...,...,...,...
145,1231da8c-d212-482f-9322-7cfb29f58ec4,Alyssa Baxter,2001,United States Virgin Islands,31
146,72112e3f-fd1a-4806-9d0a-83a20336858a,Daniel Williams,2003,Zimbabwe,114
147,a001cc34-98a5-4a4d-8c49-08e30d3e8851,Kimberly Kennedy,2004,San Marino,70
148,854b3b61-79da-4ecf-b391-f2d6b9dda491,Stephanie Payne,1979,Congo,58


# **DATA PRE-PROCESSING**

## **TẠO UNIQUE KEYS CHO DIRECTOR VÀ ACTORS**

### **DIRECTORS**

In [3]:
df_director_id = df_director.copy()

df_director_id['director_id'] = df_director_id.reset_index().index.map(
    lambda x: f"DIR{str(x+1).zfill(3)}"
)
df_director_id = df_director_id[['director_id', 'director', 'director_score']]

### **ACTORS**

In [4]:
df_actor_id = df_actor.copy()

df_actor_id['actor_id'] = df_actor_id.reset_index().index.map(
    lambda x: f"ACT{str(x+1).zfill(3)}"
)
df_actor_id = df_actor_id[['actor_id', 'actor', 'actor_score']]

In [5]:
dictionary_director_mapping = df_director_id.set_index('director')['director_id'].to_dict()
dictionary_actor_mapping = df_actor_id.set_index('actor')['actor_id'].to_dict()


In [9]:
df_movie_id = df_movie.copy()

df_movie_id['director_id'] = df_movie_id['director'].map(dictionary_director_mapping)

def map_cast_string_to_ids(cast_string, actor_map):
    names = cast_string.split(', ')
    ids = [actor_map.get(name) for name in names]
    return [id for id in ids if id is not None]

df_movie_id['cast_ids'] = df_movie_id['cast'].apply(
    lambda x: map_cast_string_to_ids(x, dictionary_actor_mapping)
)
df_movie_id = df_movie_id.drop(columns=['director', 'cast'])

In [10]:
df_movie_id

,movie_id,title,genre,budget,imdb_rating,countries_streamed,avg_watch_time,avg_rating_given,num_viewers,movie_success_score,recommended_to_buy,director_id,cast_ids
0,8a39d29d-d26c-414f-8722-180473f2db11,Triple-buffered actuating capacity,Thriller,0.64,6.82,63,107.52,3.34,632231,71.69,1,DIR030,"[ACT014, ACT026, ACT016, ACT104, ACT087]"
1,5201f3ad-a8f2-4bf6-80f4-5a4f26cb5f04,Upgradable analyzing definition,Drama,6.24,6.57,43,103.56,3.36,1027651,53.61,0,DIR013,"[ACT115, ACT036, ACT109]"
2,f07ce4dc-9226-4170-8bfa-4a254d458820,Visionary real-time info-mediaries,Animation,75.22,6.60,51,97.72,2.87,990894,71.51,1,DIR005,"[ACT026, ACT013, ACT004, ACT024]"
3,4611f68a-ae4c-4fa3-afd3-dcde85db3992,Grass-roots scalable artificial intelligence,Animation,0.91,6.04,36,103.71,2.68,139346,54.43,0,DIR011,"[ACT125, ACT124, ACT055, ACT103]"
4,15e081d0-50dc-4503-9257-8884709090d4,Horizontal heuristic toolset,Horror,35.11,6.72,66,110.74,3.04,961595,67.79,0,DIR001,"[ACT068, ACT117, ACT074, ACT109]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
79995,11eb420a-916c-4aa5-9b27-d41993fded7f,Adaptive methodical initiative,Drama,5.62,7.00,42,116.55,3.45,816365,54.79,0,DIR020,"[ACT074, ACT001, ACT130]"
79996,bc71b4ba-274a-4eb3-8d60-f9ec3c841424,Reduced human-resource Internet solution,Drama,0.52,6.22,64,96.16,3.30,219315,51.84,0,DIR023,"[ACT072, ACT008, ACT009]"
79997,e68e9300-a5b0-4adc-80dc-fee78fdbe54c,Open-architected maximized standardization,Drama,3.08,7.42,54,133.03,3.83,923642,67.97,0,DIR006,"[ACT061, ACT114, ACT105, ACT101]"
79998,f17b3c79-4bf4-42a2-b462-7533932268ae,Open-source bandwidth-monitored analyzer,Animation,0.45,6.94,46,109.87,3.72,1153861,80.10,1,DIR032,"[ACT072, ACT020, ACT065, ACT123]"


## **EXPLODE ACTOR THÀNH NHIỀU RECORDS & JOIN DIRECTORS VÀ ACTORS**

In [14]:
df_processed = pd.merge(
    df_movie_id, 
    df_director_id[['director_id', 'director_score']], 
    on='director_id', 
    how='left'
)

In [15]:
df_movie_exploded = df_processed.explode('cast_ids')
df_movie_exploded = df_movie_exploded.rename(columns={'cast_ids': 'actor_id'})


In [17]:
df_movie_exploded_with_scores = pd.merge(
    df_movie_exploded,
    df_actor_id[['actor_id', 'actor_score']],
    on='actor_id',
    how='left'
)

In [18]:
df_movie_exploded_with_scores

,movie_id,title,genre,budget,imdb_rating,countries_streamed,avg_watch_time,avg_rating_given,num_viewers,movie_success_score,recommended_to_buy,director_id,actor_id,director_score,actor_score
0,8a39d29d-d26c-414f-8722-180473f2db11,Triple-buffered actuating capacity,Thriller,0.64,6.82,63,107.52,3.34,632231,71.69,1,DIR030,ACT014,73.204932,68.986482
1,8a39d29d-d26c-414f-8722-180473f2db11,Triple-buffered actuating capacity,Thriller,0.64,6.82,63,107.52,3.34,632231,71.69,1,DIR030,ACT026,73.204932,75.880707
2,8a39d29d-d26c-414f-8722-180473f2db11,Triple-buffered actuating capacity,Thriller,0.64,6.82,63,107.52,3.34,632231,71.69,1,DIR030,ACT016,73.204932,88.999634
3,8a39d29d-d26c-414f-8722-180473f2db11,Triple-buffered actuating capacity,Thriller,0.64,6.82,63,107.52,3.34,632231,71.69,1,DIR030,ACT104,73.204932,60.667006
4,8a39d29d-d26c-414f-8722-180473f2db11,Triple-buffered actuating capacity,Thriller,0.64,6.82,63,107.52,3.34,632231,71.69,1,DIR030,ACT087,73.204932,64.019332
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320203,f17b3c79-4bf4-42a2-b462-7533932268ae,Open-source bandwidth-monitored analyzer,Animation,0.45,6.94,46,109.87,3.72,1153861,80.10,1,DIR032,ACT123,80.135248,69.973017
320204,c5931939-9d2a-4813-9ea4-6e10723c60af,Monitored mission-critical workforce,Romance,0.75,5.54,33,111.68,2.90,181893,59.43,0,DIR043,ACT043,52.534405,73.921223
320205,c5931939-9d2a-4813-9ea4-6e10723c60af,Monitored mission-critical workforce,Romance,0.75,5.54,33,111.68,2.90,181893,59.43,0,DIR043,ACT070,52.534405,72.627915
320206,c5931939-9d2a-4813-9ea4-6e10723c60af,Monitored mission-critical workforce,Romance,0.75,5.54,33,111.68,2.90,181893,59.43,0,DIR043,ACT091,52.534405,38.987403


## **TÍNH AVG SCORE CỦA ACTORS**

In [20]:
df_movie_with_cast_avg_scores = df_movie_exploded_with_scores.groupby('movie_id')['actor_score'].mean()

df_movie_with_cast_avg_scores = df_movie_with_cast_avg_scores.reset_index().rename(
    columns={'actor_score': 'cast_avg_score'}
)

In [21]:
df_movie_with_cast_avg_scores

,movie_id,cast_avg_score
0,000017b5-1c66-443a-9c9e-0369d6ee1874,75.302541
1,0000c6c1-8fe2-433f-9119-a6acdcff3903,49.396958
2,0002e809-69ca-4f8c-81c1-3f3d1fd06e5b,51.634747
3,00038639-f6f3-4968-aa65-3bea2f89f397,58.087255
4,0003d135-f90b-4400-9adf-b8ec36e1c4f3,51.420799
...,...,...
79995,fffca6c6-1ed1-4a47-a370-1f2a0c5f3f14,93.245921
79996,fffd088a-13d4-45ed-9292-8115343d555a,86.607640
79997,fffd7abc-35db-4f99-a751-a5dfbd9edb9f,52.024934
79998,fffebae4-be70-4c65-b34a-1018800419f4,55.714614


## **FINAL FEATURES**

In [25]:
df_final_features = df_processed.drop(columns=['cast_ids', 'director_id'])
df_final_features = pd.merge(
    df_final_features,
    df_movie_with_cast_avg_scores,
    on='movie_id',
    how='left'
)

feature_cols = ['genre', 'budget', 'director_score', 'cast_avg_score']
target_cols = ['recommended_to_buy', 'movie_success_score']
other_cols = [col for col in df_final_features.columns if col not in feature_cols + target_cols]

df_final_features = df_final_features[other_cols + feature_cols + target_cols]

In [26]:
df_final_features

,movie_id,title,imdb_rating,countries_streamed,avg_watch_time,avg_rating_given,num_viewers,genre,budget,director_score,cast_avg_score,recommended_to_buy,movie_success_score
0,8a39d29d-d26c-414f-8722-180473f2db11,Triple-buffered actuating capacity,6.82,63,107.52,3.34,632231,Thriller,0.64,73.204932,71.710632,1,71.69
1,5201f3ad-a8f2-4bf6-80f4-5a4f26cb5f04,Upgradable analyzing definition,6.57,43,103.56,3.36,1027651,Drama,6.24,41.459478,74.173016,0,53.61
2,f07ce4dc-9226-4170-8bfa-4a254d458820,Visionary real-time info-mediaries,6.60,51,97.72,2.87,990894,Animation,75.22,80.505917,72.678718,1,71.51
3,4611f68a-ae4c-4fa3-afd3-dcde85db3992,Grass-roots scalable artificial intelligence,6.04,36,103.71,2.68,139346,Animation,0.91,52.025089,72.772430,0,54.43
4,15e081d0-50dc-4503-9257-8884709090d4,Horizontal heuristic toolset,6.72,66,110.74,3.04,961595,Horror,35.11,75.168474,60.384153,0,67.79
...,...,...,...,...,...,...,...,...,...,...,...,...,...
79995,11eb420a-916c-4aa5-9b27-d41993fded7f,Adaptive methodical initiative,7.00,42,116.55,3.45,816365,Drama,5.62,40.357432,66.564203,0,54.79
79996,bc71b4ba-274a-4eb3-8d60-f9ec3c841424,Reduced human-resource Internet solution,6.22,64,96.16,3.30,219315,Drama,0.52,58.713778,68.798112,0,51.84
79997,e68e9300-a5b0-4adc-80dc-fee78fdbe54c,Open-architected maximized standardization,7.42,54,133.03,3.83,923642,Drama,3.08,77.218472,65.210872,0,67.97
79998,f17b3c79-4bf4-42a2-b462-7533932268ae,Open-source bandwidth-monitored analyzer,6.94,46,109.87,3.72,1153861,Animation,0.45,80.135248,61.984029,1,80.10


---

# **TRAIN MODEL**

# Dataset sẽ có 2 kịch bản để train model:
## **Kịch bản 1: "Green-light"**
- **Câu hỏi nghiệp vụ: "Tôi có nên mua bản quyền/đầu tư sản xuất bộ phim này trước khi nó ra mắt không?"**
- **Thời điểm dự đoán:** Trước khi phim phát hành.

- **Các features (Inputs):**

  - `genre` (Thể loại)

  - `budget` (Kinh phí)

  - `director_score` (Uy tín đạo diễn)

  - `cast_avg_score` (Độ nổi tiếng của dàn cast)

- **Target (Output):** `recommended_to_buy`

Trong kịch bản này, các features `avg_watch_time` và `avg_rating_given` là không thể sử dụng được. Vì tại thời điểm nhà sản xuất ra quyết định, phim chưa chiếu. Nên không thể biết thời gian xem trung bình hay điểm khán giả chấm sẽ là bao nhiêu.

Nếu đưa 2 feature này vào mô hình "Green-light", đó gọi là Rò rỉ dữ liệu (Data Leakage). Mô hình sẽ học được một mối quan hệ "gian lận" (ví dụ: "nếu khán giả chấm điểm cao thì phim sẽ thành công") mà trong thực tế không thể áp dụng được, vì không biết trước điểm khán giả chấm.

Vì vậy, 4 features (genre, budget, director_score, cast_avg_score) là hoàn toàn đủ và chính xác cho bài toán này.

## **Kịch bản 2: "Performance Tracking" (Mô hình theo dõi hiệu suất)**
- **Câu hỏi nghiệp vụ:** "Bộ phim này đã chiếu được 1 tháng. Dựa trên số liệu tháng đầu, hãy dự đoán xem tổng doanh thu (hoặc movie_success_score cuối cùng) của nó sẽ cao đến mức nào?"

- **Thời điểm dự đoán:** 1 tuần sau khi phim phát hành.

- **Các features (Inputs):**

  - Tất cả 4 features ở Kịch bản 1.

  - VÀ: `avg_watch_time` (của tháng đầu tiên).

  - VÀ: `avg_rating_given` (của tháng đầu tiên).

  - `num_viewers` (của tháng đầu tiên).

  - `imdb_rating` (hiện tại).

- **Target (Output): `movie_success_score`.**

Trong kịch bản này, `avg_watch_time` và `avg_rating_given` important. 2 fields này là những cleading indicators mạnh nhất cho sự mô hình.

## **CHỌN KỊCH BẢN**

### ***TO DO: MODEL 1***

#### ***TUNNING MODEL***

#### ***TRAIN MODEL***

#### ***ĐÁNH GIÁ MODEL***

### ***TO DO: MODEL 2***

#### ***TUNNING MODEL***

#### ***TRAIN MODEL***

#### ***ĐÁNH GIÁ MODEL***

### ***TO DO: MODEL 3***

#### ***TUNNING MODEL***

#### ***TRAIN MODEL***

#### ***ĐÁNH GIÁ MODEL***